# 02b — Reconditioned full-state flow-map baseline

Questo notebook ripete il confronto B0/B1/B3 senza cambiare dataset, split, traiettorie, input U1/U2 o backbone B3. Corregge esclusivamente normalizzazione zero-inflated, loss, early stopping e checkpoint. Non implementa HayFlow-Hines o l'architettura finale.

## 1. Checkout riproducibile

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

ELM_REPOSITORY = "https://github.com/Zagred47/giada.git"
ELM_REF = os.environ.get("HAYFLOW_ELM_REF", "main")
NOTEBOOK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd().resolve()
WORKSPACE = NOTEBOOK_ROOT / "hayflow_workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)

def run(command, cwd=None):
    print("+", " ".join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)

elm_override = os.environ.get("HAYFLOW_ELM_REPO")
mounted = [Path(elm_override).expanduser()] if elm_override else []
mounted.extend([Path.cwd(), *Path.cwd().parents])
mounted_elm = next((p.resolve() for p in mounted if (p / "src" / "hayflow_model").is_dir()), None)
if mounted_elm is not None:
    ELM_REPO = mounted_elm
else:
    ELM_REPO = WORKSPACE / "elmneuron"
    if not (ELM_REPO / ".git").is_dir():
        run(["git", "clone", ELM_REPOSITORY, ELM_REPO])
    run(["git", "fetch", "origin", ELM_REF], cwd=ELM_REPO)
    run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ELM_REPO)
os.environ["HAYFLOW_ELM_REPO"] = str(ELM_REPO)
sys.path.insert(0, str(ELM_REPO))
for module_name in tuple(sys.modules):
    if module_name == "src.hayflow_data" or module_name.startswith("src.hayflow_data.") or module_name == "src.hayflow_model" or module_name.startswith("src.hayflow_model.") or module_name == "src.hayflow_eval" or module_name.startswith("src.hayflow_eval."):
        del sys.modules[module_name]
print("Repository:", ELM_REPO)


## 2. Dipendenze e GPU

Il profilo completo richiede una GPU. `smoke` verifica soltanto il cablaggio e non produce una decisione scientifica.

In [ ]:
run([sys.executable, "-m", "pip", "install", "--quiet", "h5py", "pyarrow", "pyyaml", "pandas", "matplotlib"])
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 3. Input immutabili

Sono obbligatori sia `hayflow_diagnostic_dataset_v1.zip` sia `hayflow_full_state_flowmap_baseline.zip`. Il secondo è il risultato originale 02 e viene usato soltanto per il confronto, mai come training input.

In [ ]:
dataset_override = os.environ.get("HAYFLOW_DIAGNOSTIC_DATASET")
original_override = os.environ.get("HAYFLOW_ORIGINAL_02_RESULT")
dataset_candidates = [Path(dataset_override).expanduser()] if dataset_override else []
original_candidates = [Path(original_override).expanduser()] if original_override else []
if Path("/kaggle/input").is_dir():
    dataset_candidates.extend(Path("/kaggle/input").rglob("hayflow_diagnostic_dataset_v1.zip"))
    dataset_candidates.extend(path.parent for path in Path("/kaggle/input").rglob("dataset_manifest.json"))
    original_candidates.extend(Path("/kaggle/input").rglob("hayflow_full_state_flowmap_baseline.zip"))
    original_candidates.extend(path.parent for path in Path("/kaggle/input").rglob("experiment_manifest.json"))
DATASET_SOURCE = next((path.resolve() for path in dataset_candidates if path.exists()), None)
def is_original_02_source(path):
    path = Path(path)
    if path.is_file():
        return path.name == "hayflow_full_state_flowmap_baseline.zip"
    manifest = path / "experiment_manifest.json"
    if not manifest.is_file():
        return False
    try:
        return json.loads(manifest.read_text()).get("experiment") == "full_state_flowmap_baseline"
    except Exception:
        return False
ORIGINAL_02_SOURCE = next((path.resolve() for path in original_candidates if path.exists() and is_original_02_source(path)), None)
assert DATASET_SOURCE is not None, "Dataset diagnostico v1.0.1 non trovato."
assert ORIGINAL_02_SOURCE is not None, ("Risultato originale 02 non trovato. Aggiungi " "hayflow_full_state_flowmap_baseline.zip agli input Kaggle.")
print("Dataset:", DATASET_SOURCE)
print("Original 02:", ORIGINAL_02_SOURCE)


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def prepare_original_result(source, cache):
    source = Path(source).resolve()
    if source.is_dir():
        candidates = [source, *[p.parent for p in source.rglob("experiment_manifest.json")]]
    else:
        assert source.suffix.lower() == ".zip"
        cache = Path(cache).resolve()
        cache.mkdir(parents=True, exist_ok=True)
        marker = cache / ".source_sha256"
        source_hash = sha256_file(source)
        if not marker.is_file() or marker.read_text().strip() != source_hash:
            for child in cache.iterdir():
                if child.name != ".source_sha256":
                    raise RuntimeError("Cache original 02 non vuota e fingerprint differente; usa una nuova directory.")
            with zipfile.ZipFile(source) as archive:
                for info in archive.infolist():
                    target = (cache / info.filename).resolve()
                    assert cache == target or cache in target.parents, f"Unsafe ZIP path: {info.filename}"
                archive.extractall(cache)
            marker.write_text(source_hash + "\n")
        candidates = [p.parent for p in cache.rglob("experiment_manifest.json")]
    valid = []
    for candidate in candidates:
        manifest_path = candidate / "experiment_manifest.json"
        if manifest_path.is_file():
            manifest = json.loads(manifest_path.read_text())
            if manifest.get("experiment") == "full_state_flowmap_baseline":
                valid.append(candidate.resolve())
    assert len(set(valid)) == 1, f"Atteso un solo risultato 02, trovati: {valid}"
    return valid[0]

from src.hayflow_data import prepare_flowmap_bundle
bundle = prepare_flowmap_bundle(DATASET_SOURCE, cache_dir=WORKSPACE / "diagnostic_dataset_v1_0_1", verify_hashes=True)
ORIGINAL_02_ROOT = prepare_original_result(ORIGINAL_02_SOURCE, WORKSPACE / "original_02_result")
display({"dataset_schema": bundle.schema_version, "transition_count": bundle.manifest["transition_count"], "original_02_root": str(ORIGINAL_02_ROOT)})


## 4. Configurazione congelata e audit train-only

In [ ]:
import yaml
from src.hayflow_model import ReconditionedExperimentConfig, ReconditionedFlowmapExperiment
CONFIG_PATH = ELM_REPO / "configs" / "hayflow" / "reconditioned_full_state_flowmap_baseline.yml"
document = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
raw = dict(document["experiment"])
raw["initialization_seeds"] = tuple(raw["initialization_seeds"])
raw["rollout_horizons_ms"] = tuple(raw["rollout_horizons_ms"])
raw["profile"] = os.environ.get("HAYFLOW_02B_PROFILE", raw["profile"])
config = ReconditionedExperimentConfig(**raw)
if config.profile == "diagnostic_full" and not torch.cuda.is_available():
    raise RuntimeError("Abilita una GPU Kaggle per diagnostic_full.")
OUTPUT_DIR = ELM_REPO / "artifacts" / "reconditioned_full_state_flowmap_baseline"
experiment = ReconditionedFlowmapExperiment(bundle, ORIGINAL_02_ROOT, OUTPUT_DIR, config)
preparation = experiment.prepare()
display(preparation)


## 5. Preflight bloccante

Costruisce tutte le varianti, verifica hurdle heads, privileged heads, causalità e fingerprint prima del training.

In [ ]:
preflight = experiment.preflight()
display(preflight)
assert preflight["valid"]
assert preflight["checkpoint_fingerprint_fields_complete"]
assert not preflight["release_outcome_leakage"]


## 6. B0/B1/B3 ricondizionati

La cella è lunga. Confronta S0/S1, U1/U2, P0/P1a/P1b e tre seed comuni. L'early stopping non usa synapse o privileged loss. I checkpoint sono riutilizzati soltanto se il fingerprint completo coincide. Il report finale risponde separatamente alle sei domande scientifiche predefinite.

In [ ]:
final_report = experiment.run()
assert final_report["valid"]
display({"methodological_validity": final_report["methodological_validity"], "modeling_result": final_report["modeling_result"], "decision": final_report["decision"]})
display(final_report["scientific_questions"])


## 7. Validazione degli output

In [ ]:
required = document["outputs"]["required"]
missing = [name for name in required if not (OUTPUT_DIR / name).exists()]
assert not missing, missing
assert any((OUTPUT_DIR / "checkpoints").rglob("best_selection.pt"))
assert any((OUTPUT_DIR / "checkpoints").rglob("best_rollout.pt"))
assert any((OUTPUT_DIR / "figures").glob("*.png"))
print("Output 02b completo:", OUTPUT_DIR)


## 8. Download ZIP

Usa il trasferimento Base64/Blob richiesto per Kaggle, con un vero link temporaneo `<a download>`.

In [ ]:
import base64
from shutil import make_archive
from IPython.display import Javascript, display
archive_base = NOTEBOOK_ROOT / "hayflow_reconditioned_full_state_flowmap_baseline"
zip_path = Path(make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name))
encoded = base64.b64encode(zip_path.read_bytes()).decode("ascii")
filename = zip_path.name
display(Javascript(f"""
(() => {{
  const binary = atob('{encoded}');
  const bytes = new Uint8Array(binary.length);
  for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
  const blob = new Blob([bytes], {{type: 'application/zip'}});
  const url = URL.createObjectURL(blob);
  const anchor = document.createElement('a');
  anchor.href = url;
  anchor.download = '{filename}';
  document.body.appendChild(anchor);
  anchor.click();
  anchor.remove();
  setTimeout(() => URL.revokeObjectURL(url), 60000);
}})();
"""))
print("Download avviato:", zip_path, f"({zip_path.stat().st_size / 1e6:.1f} MB)")
